# Optimize dataset for TSP

This notebook starts from `municipalities.geojson` and `municipalityShapes.geojson` 

It cleans up the `municipalityShapes.geojson`, merging merged municipalities
It matches each municipalityShape Name to its corresponding `municipalities.geojson` feature

It calculates each municipality neighbor from municipalityShapes
It gets the actual route between each neighbor from the google maps Directions API

It stores this in `routes_with_directions.csv`

In [2]:
import json
import geopandas as gpd
import unidecode
import pandas as pd
from shapely.ops import unary_union

# import networkx as nx
# import matplotlib.pyplot as plt
# import numpy as np
# from geopy.distance import geodesic
# import folium

## Load data

municipalities.geojson contains a point for all the 581 municipalities, created in `TSPcreateDataset.ipynb`

In [3]:
# Path to the GeoJSON file
filename = 'municipalities.geojson'

# Read the GeoJSON file
with open(filename, 'r') as file:
    municipalityPoints_fc = json.load(file)

# Print the number of features
num_features = len(municipalityPoints_fc['features'])
print("Number of features:", num_features)

Number of features: 581


municipalityShapes.geojson contains the shape for 589 municipalities, downloaded from [ArcGIS Hub](https://hub.arcgis.com/datasets/9589d9e5e5904f1ea8d245b54f51b4fd/explore?location=51.006906%2C4.621885%2C9.72)

In [4]:
# Load your geographic data
municipalityShapes_df = gpd.read_file("municipalityShapes.geojson")

# number of rows
print(municipalityShapes_df.shape[0])

589


## Cleanup municipalityShapes

since the creation of the municipalityShapes dataset some municipalities have been merged.

In [5]:
print(municipalityShapes_df.head())

   OBJECTID    ADMUNAFR    ADMUNADU    ADMUNAGE    Communes CODE_INS arrond  \
0         1  AARTSELAAR  AARTSELAAR  AARTSELAAR  Aartselaar    11001     11   
1         2      ANVERS   ANTWERPEN   ANTWERPEN   Antwerpen    11002     11   
2         3    BOECHOUT    BOECHOUT    BOECHOUT    Boechout    11004     11   
3         4        BOOM        BOOM        BOOM        Boom    11005     11   
4         5    BORSBEEK    BORSBEEK    BORSBEEK    Borsbeek    11007     11   

                                            geometry  
0  POLYGON ((4.40125 51.14814, 4.40114 51.14797, ...  
1  POLYGON ((4.34109 51.35766, 4.34112 51.35760, ...  
2  POLYGON ((4.52882 51.19051, 4.52971 51.19020, ...  
3  POLYGON ((4.36411 51.10597, 4.36456 51.10596, ...  
4  POLYGON ((4.48350 51.20315, 4.48354 51.20314, ...  


In [6]:
def getNewCombinedRow(names, gdf):
    selected_gdf = gdf[gdf['ADMUNADU'].isin(names)]
    combined_geometry = unary_union(selected_gdf.geometry)
    return gpd.GeoDataFrame(geometry=[combined_geometry], crs=selected_gdf.crs)

def getNewlyMerged(mergers, gdf):
    # Prepare a list to hold each modified GeoDataFrame row
    modified_rows = []

    # Process each merge instruction set
    for merge in mergers:
        names, target_name, new_admunaf, new_admunad, new_admunag, new_commune = merge
        # Get the new combined geometry row from your custom function
        new_gdf = getNewCombinedRow(names, gdf)  # Assuming this function returns a GeoDataFrame

        # Fetch the row to be updated
        pelt_df = gdf.loc[gdf['ADMUNADU'] == target_name].copy().reset_index(drop=True)
        
        # Update the attributes as specified
        pelt_df.at[0, 'ADMUNAFR'] = new_admunaf
        pelt_df.at[0, 'ADMUNADU'] = new_admunad
        pelt_df.at[0, 'ADMUNAGE'] = new_admunag
        pelt_df.at[0, 'Communes'] = new_commune
        pelt_df.at[0, 'geometry'] = new_gdf.at[0, 'geometry']

        # Append the updated DataFrame to the list
        modified_rows.append(pelt_df)

    # Concatenate all modified rows into a single GeoDataFrame
    newlyMerged_df = pd.concat(modified_rows, ignore_index=True)
    return newlyMerged_df

In [7]:
mergers = [
    [['LOVENDEGEM','ZOMERGEM','WAARSCHOOT'],'WAARSCHOOT','LIEVEGEM','LIEVEGEM','LIEVEGEM','Lievegem'],
    [['OVERPELT','NEERPELT'],'OVERPELT','PELT','PELT','PELT','Pelt'],
    [['PUURS','SINT-AMANDS'],'PUURS','PUURS-SINT-AMANDS','PUURS-SINT-AMANDS','PUURS-SINT-AMANDS','Puurs-Sint-Amands'],
    [['AALTER','KNESSELARE'],'AALTER','AALTER','AALTER','AALTER','Aalter'],
    [['DEINZE','NEVELE'],'DEINZE','DEINZE','DEINZE','DEINZE','Deinze'],
    [['MEEUWEN-GRUITRODE','OPGLABBEEK'],'MEEUWEN-GRUITRODE','OUDSBERGEN','OUDSBERGEN','OUDSBERGEN','Oudsbergen'],
    [['KRUISHOUTEM','ZINGEM'],'KRUISHOUTEM','KRUISEM','KRUISEM','KRUISEM','Kruisem']
]

newlyMerged_df = getNewlyMerged(mergers, municipalityShapes_df)

In [8]:
mergedShapes_df = municipalityShapes_df

In [9]:
deprecatedMunicipalityNames = ['OVERPELT','NEERPELT','LOVENDEGEM','ZOMERGEM','WAARSCHOOT','PUURS','SINT-AMANDS', 'KNESSELARE', 'NEVELE', 'MEEUWEN-GRUITRODE','OPGLABBEEK','KRUISHOUTEM','ZINGEM','DEINZE','AALTER']

In [10]:
mergedShapes_df = mergedShapes_df[~mergedShapes_df['ADMUNADU'].isin(deprecatedMunicipalityNames)]
mergedShapes_df = pd.concat([mergedShapes_df, newlyMerged_df], ignore_index=True)

In [11]:
print(mergedShapes_df.info())

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 581 entries, 0 to 580
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype   
---  ------    --------------  -----   
 0   OBJECTID  581 non-null    int64   
 1   ADMUNAFR  581 non-null    object  
 2   ADMUNADU  581 non-null    object  
 3   ADMUNAGE  581 non-null    object  
 4   Communes  581 non-null    object  
 5   CODE_INS  581 non-null    object  
 6   arrond    581 non-null    object  
 7   geometry  581 non-null    geometry
dtypes: geometry(1), int64(1), object(6)
memory usage: 36.4+ KB
None


Okay we have 2 municipalities too much

In [12]:
mergedShapeNames = mergedShapes_df['ADMUNADU'].tolist()
print(mergedShapeNames)
print(len(mergedShapeNames))

['AARTSELAAR', 'ANTWERPEN', 'BOECHOUT', 'BOOM', 'BORSBEEK', 'BRASSCHAAT', 'BRECHT', 'EDEGEM', 'ESSEN', 'HEMIKSEM', 'HOVE', 'KALMTHOUT', 'KAPELLEN', 'KONTICH', 'LINT', 'MORTSEL', 'NIEL', 'RANST', 'RUMST', 'SCHELLE', 'SCHILDE', 'SCHOTEN', 'STABROEK', 'WIJNEGEM', 'WOMMELGEM', 'WUUSTWEZEL', 'ZANDHOVEN', 'ZOERSEL', 'ZWIJNDRECHT', 'MALLE', 'BERLAAR', 'BONHEIDEN', 'BORNEM', 'DUFFEL', 'HEIST-OP-DEN-BERG', 'LIER', 'MECHELEN', 'NIJLEN', 'PUTTE', 'SINT-KATELIJNE-WAVER', 'WILLEBROEK', 'ARENDONK', 'BAARLE-HERTOG', 'BALEN', 'BEERSE', 'DESSEL', 'GEEL', 'GROBBENDONK', 'HERENTALS', 'HERENTHOUT', 'HERSELT', 'HOOGSTRATEN', 'HULSHOUT', 'KASTERLEE', 'LILLE', 'MEERHOUT', 'MERKSPLAS', 'MOL', 'OLEN', 'OUD-TURNHOUT', 'RAVELS', 'RETIE', 'RIJKEVORSEL', 'TURNHOUT', 'VORSELAAR', 'VOSSELAAR', 'WESTERLO', 'LAAKDAL', 'ANDERLECHT', 'OUDERGHEM', 'SINT-AGATHA-BERCHEM', 'BRUSSEL', 'ETTERBEEK', 'EVERE', 'VORST', 'GANSHOREN', 'ELSENE', 'JETTE', 'KOEKELBERG', 'SINT-JANS-MOLENBEEK', 'SINT-GILLIS', 'SINT-JOOST-TEN-NODE', 'SCH

In [13]:
features = municipalityPoints_fc['features']
municipalityPointNames = [feature['properties']['name'] for feature in features if 'name' in feature['properties']]

print(municipalityPointNames)
print(len(municipalityPointNames))

['Charleroi', 'Oud-Heverlee', 'Maasmechelen', 'Saint-Gilles - Sint-Gillis', 'Etterbeek', 'Mortsel', 'Borsbeek', 'Andenne', 'Éghezée', 'Marche-en-Famenne', 'Yvoir', "'s-Gravenvoeren - Fouron-le-Comte", 'Lier', 'Nijlen', 'Heist-op-den-Berg', 'Lessines', 'Herenthout', 'Lendelede', 'Turnhout', 'Berlaar', 'Kapelle-op-den-Bos', 'Staden', 'Bonheiden', 'Ath', 'Sint-Katelijne-Waver', 'Flobecq - Vloesberg', 'Martelange', 'Leuze-en-Hainaut', 'Tournai', 'Arendonk', 'Assesse', 'Pecq', 'Beveren', 'Genappe', 'Zaventem', 'Kortenberg', 'Bertem', 'Hooglede', 'Wavre', 'Nivelles', 'Huy', 'Gembloux', 'Couvin', 'Philippeville', 'Hasselt', 'Tubize', 'Ellezelles', 'Moerbeke', 'Enghien - Edingen', 'Frasnes-lez-Buissenal', 'Arlon', 'Kalmthout', 'Aarschot', 'Beaumont', 'Beringen', 'Bilzen', 'Blankenberge', 'Bouillon', 'Bree', 'Châtelet', 'Chièvres', 'Chimay', 'Chiny', 'Ciney', 'Damme', 'Deinze', 'Diest', 'Eeklo', 'Eupen', 'Fleurus', 'Florenville', "Fontaine-l'Évêque", 'Fosses-la-Ville', 'Gistel', 'Halen', 'Hannu

In [14]:
municipalityShapes_df = mergedShapes_df

In [15]:
writeToFile = False
if writeToFile:
    municipalityShapes_df.to_file("municipalityMerge.geojson", driver='GeoJSON')

## find matches

We want to calculate the actual distance between two villages

We only need to calculate the neighboring villages

maybe we need extra villages for the bordering villages

we can get the neighbors from municipalityShapes_df 

but first we have to match up the dataset and merge some features

In [16]:
features = municipalityPoints_fc['features']
municipalityPointNames = [feature['properties']['name'] for feature in features if 'name' in feature['properties']]

print(len(municipalityPointNames))
print(municipalityPointNames)

581
['Charleroi', 'Oud-Heverlee', 'Maasmechelen', 'Saint-Gilles - Sint-Gillis', 'Etterbeek', 'Mortsel', 'Borsbeek', 'Andenne', 'Éghezée', 'Marche-en-Famenne', 'Yvoir', "'s-Gravenvoeren - Fouron-le-Comte", 'Lier', 'Nijlen', 'Heist-op-den-Berg', 'Lessines', 'Herenthout', 'Lendelede', 'Turnhout', 'Berlaar', 'Kapelle-op-den-Bos', 'Staden', 'Bonheiden', 'Ath', 'Sint-Katelijne-Waver', 'Flobecq - Vloesberg', 'Martelange', 'Leuze-en-Hainaut', 'Tournai', 'Arendonk', 'Assesse', 'Pecq', 'Beveren', 'Genappe', 'Zaventem', 'Kortenberg', 'Bertem', 'Hooglede', 'Wavre', 'Nivelles', 'Huy', 'Gembloux', 'Couvin', 'Philippeville', 'Hasselt', 'Tubize', 'Ellezelles', 'Moerbeke', 'Enghien - Edingen', 'Frasnes-lez-Buissenal', 'Arlon', 'Kalmthout', 'Aarschot', 'Beaumont', 'Beringen', 'Bilzen', 'Blankenberge', 'Bouillon', 'Bree', 'Châtelet', 'Chièvres', 'Chimay', 'Chiny', 'Ciney', 'Damme', 'Deinze', 'Diest', 'Eeklo', 'Eupen', 'Fleurus', 'Florenville', "Fontaine-l'Évêque", 'Fosses-la-Ville', 'Gistel', 'Halen', 'H

In [17]:
municipalityShapeNames = municipalityShapes_df['ADMUNADU'].tolist()
print(len(municipalityShapeNames))
print(municipalityShapeNames)

581
['AARTSELAAR', 'ANTWERPEN', 'BOECHOUT', 'BOOM', 'BORSBEEK', 'BRASSCHAAT', 'BRECHT', 'EDEGEM', 'ESSEN', 'HEMIKSEM', 'HOVE', 'KALMTHOUT', 'KAPELLEN', 'KONTICH', 'LINT', 'MORTSEL', 'NIEL', 'RANST', 'RUMST', 'SCHELLE', 'SCHILDE', 'SCHOTEN', 'STABROEK', 'WIJNEGEM', 'WOMMELGEM', 'WUUSTWEZEL', 'ZANDHOVEN', 'ZOERSEL', 'ZWIJNDRECHT', 'MALLE', 'BERLAAR', 'BONHEIDEN', 'BORNEM', 'DUFFEL', 'HEIST-OP-DEN-BERG', 'LIER', 'MECHELEN', 'NIJLEN', 'PUTTE', 'SINT-KATELIJNE-WAVER', 'WILLEBROEK', 'ARENDONK', 'BAARLE-HERTOG', 'BALEN', 'BEERSE', 'DESSEL', 'GEEL', 'GROBBENDONK', 'HERENTALS', 'HERENTHOUT', 'HERSELT', 'HOOGSTRATEN', 'HULSHOUT', 'KASTERLEE', 'LILLE', 'MEERHOUT', 'MERKSPLAS', 'MOL', 'OLEN', 'OUD-TURNHOUT', 'RAVELS', 'RETIE', 'RIJKEVORSEL', 'TURNHOUT', 'VORSELAAR', 'VOSSELAAR', 'WESTERLO', 'LAAKDAL', 'ANDERLECHT', 'OUDERGHEM', 'SINT-AGATHA-BERCHEM', 'BRUSSEL', 'ETTERBEEK', 'EVERE', 'VORST', 'GANSHOREN', 'ELSENE', 'JETTE', 'KOEKELBERG', 'SINT-JANS-MOLENBEEK', 'SINT-GILLIS', 'SINT-JOOST-TEN-NODE', 

## match datasets

### match municipalityShapes and municipalities by ADMUNADU

In [18]:
def normalize_name(name):
    if pd.isna(name):
        return ""  # Handle NaN values
    return unidecode.unidecode(name.lower().replace("'", ""))

In [19]:
municipalityShapes_df['Normalized_Name'] = municipalityShapes_df['ADMUNADU'].apply(normalize_name)
features_df = pd.DataFrame({'Original_Name': municipalityPointNames, 'Normalized_Name': [normalize_name(name) for name in municipalityPointNames]})

# Merge DataFrames on the normalized names
merged_df = pd.merge(features_df, municipalityShapes_df[['ADMUNADU', 'Normalized_Name']], on='Normalized_Name', how='left')

In [20]:
# DataFrame with missing values in 'ADMUNADU'
df_missing = merged_df[merged_df['ADMUNADU'].isna()].copy()

# DataFrame without missing values in 'ADMUNADU'
df_found = merged_df[merged_df['ADMUNADU'].notna()]

# Optionally, print the number of rows in each DataFrame to confirm the split
print("Rows with missing 'ADMUNADU':", len(df_missing))
print("Rows without missing 'ADMUNADU':", len(df_found))

Rows with missing 'ADMUNADU': 66
Rows without missing 'ADMUNADU': 515


### match municipalityShapes and municipalities by ADMUNAFR

In [21]:
municipalityShapes_df['Normalized_ADMUNAFR'] = municipalityShapes_df['ADMUNAFR'].apply(normalize_name)
df_missing['Normalized_Original_Name'] = df_missing['Original_Name'].apply(normalize_name)

# Create a mapping function to find matches
def find_match(normalized_name, df):
    # Loop through the DataFrame passed in
    for idx, row in df.iterrows():
        if normalized_name == row['Normalized_ADMUNAFR']:
            return row['ADMUNADU']
    return None  # Return None if no match is found

# Apply the mapping function
df_missing['ADMUNADU'] = df_missing['Normalized_Original_Name'].apply(lambda x: find_match(x, municipalityShapes_df[['ADMUNADU', 'Normalized_ADMUNAFR']]))

# Select relevant columns for the final DataFrame
df_matched_french = df_missing[['Original_Name', 'ADMUNADU']]

In [22]:
# DataFrame with missing values in 'ADMUNADU'
df_missing1 = df_matched_french[df_matched_french['ADMUNADU'].isna()]

# DataFrame without missing values in 'ADMUNADU'
df_found1 = df_matched_french[df_matched_french['ADMUNADU'].notna()]

print("Rows without missing 'ADMUNADU':", len(df_found))
print("New rows without missing 'ADMUNADU':", len(df_found1))
print("Rows with missing 'ADMUNADU':", len(df_missing1))

Rows without missing 'ADMUNADU': 515
New rows without missing 'ADMUNADU': 36
Rows with missing 'ADMUNADU': 30


### Merge by part of string

In [23]:
missingNames = df_missing1['Original_Name'].tolist()
admunaduNames = municipalityShapes_df['ADMUNADU'].tolist()

In [24]:
def normalize_name(name):
    if pd.isna(name):
        return ""  # Handle NaN values
    return unidecode.unidecode(name.lower().replace("'", ""))

# Normalize the names in both lists
normalized_missing_names = [normalize_name(name) for name in missingNames]
normalized_admunadu_names = [normalize_name(name) for name in admunaduNames]

# Prepare the DataFrame
matching_data = []

# Loop through each name in missingNames along with its normalized version
for orig_missing, norm_missing in zip(missingNames, normalized_missing_names):
    match_found = None
    # Compare against all normalized ADMUNADU names
    for orig_admunadu, norm_admunadu in zip(admunaduNames, normalized_admunadu_names):
        if norm_admunadu in norm_missing:  # Check if ADMUNADU name is contained within the missing name
            match_found = orig_admunadu
            break
    # Append the data to the list
    matching_data.append({
        'Original_Name': orig_missing,
        'Normalized_Name': norm_missing,
        'ADMUNADU': match_found
    })

# Create the DataFrame
matched_df = pd.DataFrame(matching_data)


In [25]:
matched_df = matched_df.drop(columns=['Normalized_Name'])

# DataFrame with missing values in 'ADMUNADU'
df_missing2 = matched_df[matched_df['ADMUNADU'].isna()]

# DataFrame without missing values in 'ADMUNADU'
df_found2 = matched_df[matched_df['ADMUNADU'].notna()]

# Optionally, print the number of rows in each DataFrame to confirm the split
print("Rows with missing 'ADMUNADU':", len(df_missing2))
print("Rows without missing 'ADMUNADU':", len(df_found2))

Rows with missing 'ADMUNADU': 3
Rows without missing 'ADMUNADU': 27


### fix wrong matches

In [26]:
df_found2

,Original_Name,ADMUNADU
0,Saint-Gilles - Sint-Gillis,SINT-GILLIS
1,'s-Gravenvoeren - Fouron-le-Comte,GRAVEN
2,Flobecq - Vloesberg,VLOESBERG
3,Enghien - Edingen,EDINGEN
4,Frasnes-lez-Buissenal,AS
5,Saint-Josse-ten-Noode - Sint-Joost-ten-Node,SINT-JOOST-TEN-NODE
6,Schaerbeek - Schaarbeek,SCHAARBEEK
7,Berchem-Sainte-Agathe - Sint-Agatha-Berchem,SINT-AGATHA-BERCHEM
8,Molenbeek-Saint-Jean - Sint-Jans-Molenbeek,MOL
9,Uccle - Ukkel,UKKEL


In [27]:
missers = ['GRAVEN','AS','MOL','BRECHT','ASSE','BERGEN']

# the df_not_missing2 with one of these values in its ADMUNADU column should be removed, value cleared and added to df_missing2 

# Filter out rows where 'ADMUNADU' is in the list 'missers'
to_remove = df_found2['ADMUNADU'].isin(missers)
df_to_move = df_found2[to_remove].copy()

# Clear the 'ADMUNADU' values in these rows
df_to_move['ADMUNADU'] = pd.NA

# Now, remove these rows from 'df_not_missing2'
df_found2 = df_found2[~to_remove]

# Append these rows to 'df_missing2'
df_missing2 = pd.concat([df_missing2, df_to_move], ignore_index=True)

print("1st matches:", len(df_found))
print("2nd matches:", len(df_found1))
print("new matches:", len(df_found2))
print("Rows missing:", len(df_missing2))

1st matches: 515
2nd matches: 36
new matches: 22
Rows missing: 8


match the last ones manually

In [28]:
print(df_missing2)

                                   Original_Name ADMUNADU
0                           Auderghem - Oudergem     None
1                                      Büllingen     None
2                                      Thimister     None
3              's-Gravenvoeren - Fouron-le-Comte      NaN
4                          Frasnes-lez-Buissenal      NaN
5     Molenbeek-Saint-Jean - Sint-Jans-Molenbeek      NaN
6  Woluwe-Saint-Lambert - Sint-Lambrechts-Woluwe      NaN
7                              Hastière-par-delà      NaN


In [29]:
df_missing2.loc[df_missing2['Original_Name'] == 'Auderghem - Oudergem' , 'ADMUNADU'] = 'OUDERGHEM'
df_missing2.loc[df_missing2['Original_Name'] == 'Büllingen' , 'ADMUNADU'] = 'BUELLINGEN'
df_missing2.loc[df_missing2['Original_Name'] == 'Thimister' , 'ADMUNADU'] = 'THIMISTER-CLERMONT'
df_missing2.loc[df_missing2['Original_Name'] == "'s-Gravenvoeren - Fouron-le-Comte" , 'ADMUNADU'] = 'VOEREN'
df_missing2.loc[df_missing2['Original_Name'] == 'Frasnes-lez-Buissenal' , 'ADMUNADU'] = 'FRASNES-LEZ-ANVAING'
df_missing2.loc[df_missing2['Original_Name'] == 'Molenbeek-Saint-Jean - Sint-Jans-Molenbeek' , 'ADMUNADU'] = 'SINT-JANS-MOLENBEEK'
df_missing2.loc[df_missing2['Original_Name'] == 'Woluwe-Saint-Lambert - Sint-Lambrechts-Woluwe' , 'ADMUNADU'] = 'SINT-LAMBRECHTS-WOLUWE'
df_missing2.loc[df_missing2['Original_Name'] == 'Hastière-par-delà' , 'ADMUNADU'] = 'HASTIERE'

### join found names

In [30]:
df_found = df_found.drop(columns=['Normalized_Name'])

total_matched_df = pd.concat([df_found, df_found1,df_found2,df_missing2], ignore_index=True)

print(total_matched_df.head())

  Original_Name      ADMUNADU
0     Charleroi     CHARLEROI
1  Oud-Heverlee  OUD-HEVERLEE
2  Maasmechelen  MAASMECHELEN
3     Etterbeek     ETTERBEEK
4       Mortsel       MORTSEL


## Determine neighbors

We now have a match between each geojson feature and each villages neighbors

we can now find the distance between each feature and it's neighbors

todo: 

- make sure to only calculate the distance one way
- create distances / weights datatable for graph construction
- for all edges that are not neigbhors, maybe set a distance of 100km
    - caution: could cause issues for literal edge cases

In [31]:
municipalityShapes_df['neighbors'] = None  # Add a column for neighbors
for index, municipality in municipalityShapes_df.iterrows():
    # Neighbors touching the current municipality's borders
    neighbors = municipalityShapes_df[municipalityShapes_df.geometry.touches(municipality['geometry'])].ADMUNADU.tolist()
    neighbors = [name for name in neighbors if name != municipality.ADMUNADU]
    municipalityShapes_df.at[index, 'neighbors'] = ", ".join(neighbors)
    
print(municipalityShapes_df.iloc[0])

OBJECTID                                                               1
ADMUNAFR                                                      AARTSELAAR
ADMUNADU                                                      AARTSELAAR
ADMUNAGE                                                      AARTSELAAR
Communes                                                      Aartselaar
CODE_INS                                                           11001
arrond                                                                11
geometry               POLYGON ((4.4012476576079 51.1481369539823, 4....
Normalized_Name                                               aartselaar
Normalized_ADMUNAFR                                           aartselaar
neighbors              ANTWERPEN, EDEGEM, HEMIKSEM, KONTICH, NIEL, RU...
Name: 0, dtype: object


### Add the ADMUNADU names to the municipalityPoints_fc

In [32]:
# Creating a dictionary from the DataFrame for quick lookup
adm_mapping = dict(zip(total_matched_df['Original_Name'], total_matched_df['ADMUNADU']))


# Update each feature with the new 'ADMUNADU' property
for feature in municipalityPoints_fc['features']:
    original_name = feature['properties'].get('name')  # Assuming the name is stored under 'name'
    if original_name in adm_mapping:
        feature['properties']['ADMUNADU'] = adm_mapping[original_name]
    else:
        feature['properties']['ADMUNADU'] = None  # Or you can choose to not add the 'ADMUNADU' key at all

In [33]:
print(municipalityPoints_fc['features'][1:2])

[{'type': 'Feature', 'properties': {'name': 'Oud-Heverlee', 'place': 'town', 'ADMUNADU': 'OUD-HEVERLEE'}, 'geometry': {'type': 'Point', 'coordinates': [4.6629253, 50.8376275]}}]


In [34]:
print(municipalityShapes_df[['ADMUNADU','neighbors']].iloc[0])

ADMUNADU                                            AARTSELAAR
neighbors    ANTWERPEN, EDEGEM, HEMIKSEM, KONTICH, NIEL, RU...
Name: 0, dtype: object


In [68]:
def writeToFile(data, filename):
    with open(filename, 'w') as file:
        json.dump(data, file)
    print(f"Data saved to {filename}")
    
toWriteOrnotToWrite = False
if toWriteOrnotToWrite:
    writeToFile(municipalityPoints_fc, 'municipalitiesWithADMUNADU.geojson')

Data saved to municipalitiesWithADMUNADU.geojson


### Create a table of all neighboring municipalities

In [35]:
municipalityShapes_df['neighborsList'] = municipalityShapes_df['neighbors'].apply(lambda x: x.split(', ') if pd.notna(x) else [])

def calculate_distance(start, end):
    # Placeholder function to represent distance calculation
    # This should be replaced with your actual distance calculation logic
    return 42  # Replace with actual distance calculation

# Prepare to collect all rows for the new DataFrame
data = []

# Ensure each connection is calculated only once
seen_pairs = set()

for idx, row in municipalityShapes_df.iterrows():
    start = row['ADMUNADU']
    neighbors = row['neighborsList']
    for end in neighbors:
        if start and end and (end, start) not in seen_pairs:
            # Calculate the distance
            #distance = calculate_distance(start, end)
            
            # Add to data list
            #data.append({'start': start, 'end': end, 'distance': distance})
            data.append({'start': start, 'end': end})
            
            # Mark this pair as seen
            seen_pairs.add((start, end))

# Create the DataFrame
# distance_df = pd.DataFrame(data, columns=['start', 'end', 'distance'])
routes_df = pd.DataFrame(data, columns=['start', 'end'])

In [36]:
print(routes_df.head())

        start        end
0  AARTSELAAR  ANTWERPEN
1  AARTSELAAR     EDEGEM
2  AARTSELAAR   HEMIKSEM
3  AARTSELAAR    KONTICH
4  AARTSELAAR       NIEL


### Add coordinates

In [37]:
features = municipalityPoints_fc['features']
coords_dict = {}

for feature in features:
    admunadu_name = feature['properties']['ADMUNADU']
    if 'geometry' in feature and 'coordinates' in feature['geometry']:
        # Assuming points, for polygons or lines, you might need to handle it differently
        coords = feature['geometry']['coordinates']
        coords_dict[admunadu_name] = coords

# coords_dict now contains ADMUNADU names as keys and coordinates as values
coords_df = pd.DataFrame(list(coords_dict.items()), columns=['ADMUNADU', 'Coordinates'])

In [38]:
print(coords_df.head())

       ADMUNADU              Coordinates
0     CHARLEROI   [4.444528, 50.4116233]
1  OUD-HEVERLEE  [4.6629253, 50.8376275]
2  MAASMECHELEN   [5.696445, 50.9635024]
3   SINT-GILLIS  [4.3454841, 50.8249958]
4     ETTERBEEK  [4.3861737, 50.8361447]


In [74]:
coords_df[coords_df['ADMUNADU'] =='CELLES']

,ADMUNADU,Coordinates
327,CELLES,"[5.0058339, 50.2312617]"


In [39]:
# Merge to get start coordinates
routes_with_coords = routes_df.merge(coords_df, left_on='start', right_on='ADMUNADU', how='left')
routes_with_coords.rename(columns={'Coordinates': 'coords_start'}, inplace=True)
routes_with_coords.drop(columns=['ADMUNADU'], inplace=True)  # Drop the duplicate column after renaming

# Merge to get end coordinates
routes_with_coords = routes_with_coords.merge(coords_df, left_on='end', right_on='ADMUNADU', how='left')
routes_with_coords.rename(columns={'Coordinates': 'coords_end'}, inplace=True)
routes_with_coords.drop(columns=['ADMUNADU'], inplace=True)  # Drop the duplicate column after renaming

In [40]:
print(routes_with_coords.head())

        start        end            coords_start               coords_end
0  AARTSELAAR  ANTWERPEN  [4.3870241, 51.133297]  [4.3997081, 51.2211097]
1  AARTSELAAR     EDEGEM  [4.3870241, 51.133297]  [4.4458312, 51.1548066]
2  AARTSELAAR   HEMIKSEM  [4.3870241, 51.133297]  [4.3409826, 51.1430192]
3  AARTSELAAR    KONTICH  [4.3870241, 51.133297]  [4.4454784, 51.1353297]
4  AARTSELAAR       NIEL  [4.3870241, 51.133297]  [4.3303396, 51.1098649]


In [41]:
print('There are ' + str(routes_with_coords.shape[0]) + ' routes')

There are 1623 routes


In [73]:
routes_with_coords[routes_with_coords['start'] == 'CELLES']

,start,end,coords_start,coords_end,Directions,Distances
1060,CELLES,PECQ,"[5.0058339, 50.2312617]","[3.3407492, 50.685616]",[{'bounds': {'northeast': {'lat': 50.685496699...,147019
1061,CELLES,DOORNIK,"[5.0058339, 50.2312617]","[3.3878179, 50.6056458]","[{'bounds': {'northeast': {'lat': 50.6077595, ...",136969
1062,CELLES,MONT-DE-L'ENCLUS,"[5.0058339, 50.2312617]","[3.5110542, 50.7456511]","[{'bounds': {'northeast': {'lat': 50.7505596, ...",139114


## Get routes from google maps API

We have municipalityPoints_fc which now also contains the proper ADMUNADU names

We have a routes_with_coords dataframe which contains all the paths between the municipalities we have to gather

so we can use this table for graph construction

In [42]:
import googlemaps

with open('key.txt', 'r') as file:
    key = file.read().strip()

# Initialize the client with your API key
gmaps = googlemaps.Client(key = key)

/Users/tijsvandenheuvel/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


### Test API & check out directions data

In [43]:
origin = (routes_with_coords.iloc[0]['coords_start'][1], routes_with_coords.iloc[0]['coords_start'][0])
destination = (routes_with_coords.iloc[0]['coords_end'][1], routes_with_coords.iloc[0]['coords_end'][0]) 

print(origin, destination)

directions = gmaps.directions(origin, destination, mode="walking", units="metric")
print(directions)

(51.133297, 4.3870241) (51.2211097, 4.3997081)
[{'bounds': {'northeast': {'lat': 51.22104830000001, 'lng': 4.4005361}, 'southwest': {'lat': 51.1334287, 'lng': 4.378954}}, 'copyrights': 'Map data ©2024', 'legs': [{'distance': {'text': '10.5 km', 'value': 10524}, 'duration': {'text': '2 hours 25 mins', 'value': 8728}, 'end_address': 'Grote Markt 9, 2000 Antwerpen, Belgium', 'end_location': {'lat': 51.22104830000001, 'lng': 4.399776900000001}, 'start_address': 'Carillolei 2, 2630 Aartselaar, Belgium', 'start_location': {'lat': 51.1334287, 'lng': 4.3868373}, 'steps': [{'distance': {'text': '63 m', 'value': 63}, 'duration': {'text': '1 min', 'value': 54}, 'end_location': {'lat': 51.133943, 'lng': 4.3873141}, 'html_instructions': 'Head <b>northeast</b> on <b>Carillolei</b> toward <b>Laar</b>', 'polyline': {'points': '}~awHwxwYWk@EIGGMGg@UIA'}, 'start_location': {'lat': 51.1334287, 'lng': 4.3868373}, 'travel_mode': 'WALKING'}, {'distance': {'text': '11 m', 'value': 11}, 'duration': {'text': '

In [44]:
total_distance = 0
if directions:
    for leg in directions[0]['legs']:
        total_distance += leg['distance']['value']  # distance is in meters

print(f"Total Distance between {directions[0]['legs'][0]['start_address']} and {directions[0]['legs'][0]['end_address']}: {total_distance / 1000} km")  # Convert meters to kilometers

Total Distance between Carillolei 2, 2630 Aartselaar, Belgium and Grote Markt 9, 2000 Antwerpen, Belgium: 10.524 km


### Show on map

In [45]:
import folium
# Extract the polyline of the first route
polyline = directions[0]['overview_polyline']['points']

# Decode polyline (Folium can't use the encoded polyline directly)
decoded_path = googlemaps.convert.decode_polyline(polyline)

# Convert path from dictionaries to tuples of (latitude, longitude)
path = [(point['lat'], point['lng']) for point in decoded_path]

# Create a map object centered and zoomed on the first coordinate of the path
map = folium.Map(location=path[0], zoom_start=11)

# Add a polyline to the map with the decoded path coordinates
folium.PolyLine(path).add_to(map)

# Optionally, add markers for the start and end of the route
# Ensuring popup strings are only strings
folium.Marker(location=path[0], popup=f'Start: {origin}', icon=folium.Icon(color='green')).add_to(map)
folium.Marker(location=path[-1], popup=f'End: {destination}', icon=folium.Icon(color='red')).add_to(map)

# Display the map
map

### Get the data in batches

In [46]:
routes_with_coords['Directions'] = None

In [47]:
def get_directions(row):
    origin = (row['coords_start'][1], row['coords_start'][0])
    destination = (row['coords_end'][1], row['coords_end'][0])
    
    directions = gmaps.directions(origin, destination, mode="walking", units="metric")
    return directions

In [48]:
def process_dataframe(df, chunk_size=100):
    # Find indices of None values in 'Directions' column
    none_indices = df[df['Directions'].isna()].index[:chunk_size]
    
    # Fill the chunk of None values
    for idx in none_indices:
        df.at[idx, 'Directions'] = get_directions(df.loc[idx])
    
    return df

todo make this code run 17 times by itself waiting for each loop to finish

In [59]:
import time

none_counts = routes_with_coords['Directions'].isna().sum()
while none_counts !=0:
    routes_with_coords = process_dataframe(routes_with_coords)
    
    none_counts = routes_with_coords['Directions'].isna().sum()
    print(none_counts)
    
    # Wait for 7 seconds
    time.sleep(7)

900
800
700
600
500
400
300
200
100
0


In [60]:
none_counts = routes_with_coords['Directions'].isna().sum()
print(none_counts)

0


In [64]:
def extract_distance(directions_obj):
    try:
        # Assuming the structure of directions_obj is as shown in the get_directions function
        distance_value = directions_obj[0]['legs'][0]['distance']['value']
        return distance_value
    except (TypeError, KeyError, IndexError):
        return None

routes_with_coords['Distances'] = routes_with_coords['Directions'].apply(extract_distance)

In [65]:
print(routes_with_coords.loc[0])

start                                                  AARTSELAAR
end                                                     ANTWERPEN
coords_start                               [4.3870241, 51.133297]
coords_end                                [4.3997081, 51.2211097]
Directions      [{'bounds': {'northeast': {'lat': 51.221048300...
Distances                                                   10524
Name: 0, dtype: object


In [66]:
# Save the DataFrame to a CSV file
# routes_with_coords.to_csv('routes_with_directions.csv', index=False)